In [10]:
import pandas as pd

df = pd.read_csv('../data/processed/task5/train_clean_5_1.csv')

print("Dataset Shape:", df.shape)
print("\nListing Type Distribution:")
print(df['listing_type'].value_counts(dropna=False))

Dataset Shape: (5109, 66)

Listing Type Distribution:
listing_type
phone            4732
unclassified      366
parts               6
wanted              2
feature_phone       2
tablet              1
Name: count, dtype: int64


In [11]:
features = [
    'model_key', 'brand', 'family', 'storage_gb', 'ram_gb',
    'condition', 'has_5g', 'has_warranty', 'has_box',
    'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint',
    'is_tablet_hint', 'is_parts_hint', 'accessory_first',
    'bundle_hint', 'is_feature_phone_hint', 'loc_area',
    'loc_city', 'storage_known', 'ram_known', 'condition_known'
]

missing_features = [col for col in features if col not in df.columns]

print("Expected Features:", len(features))
print("Missing Features:", missing_features)
print("Target Available:", 'price' in df.columns)

Expected Features: 22
Missing Features: []
Target Available: True


In [15]:
from sklearn.model_selection import GroupShuffleSplit

# First split: 70% train, 30% temporary
splitter_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(
    splitter_1.split(df, groups=df['title_group'])
)

# Second split: 15% validation, 15% test
temp_df = df.iloc[temp_idx]

splitter_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_rel_idx, test_rel_idx = next(
    splitter_2.split(temp_df, groups=temp_df['title_group'])
)

val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

# Keep the same phone-only validation and test sets
val_df = df.iloc[val_idx]
test_df = df.iloc[test_idx]

val_phone = val_df[val_df['listing_type'] == 'phone'].copy()
test_phone = test_df[test_df['listing_type'] == 'phone'].copy()

print("Training partition:", len(train_idx))
print("Validation partition:", len(val_idx))
print("Test partition:", len(test_idx))

print("\nPhone Validation:", len(val_phone))
print("Phone Test:", len(test_phone))

print("\nGroup Overlap Check:")
print("Train-Val:", len(set(df.iloc[train_idx]['title_group']) & set(val_df['title_group'])))
print("Train-Test:", len(set(df.iloc[train_idx]['title_group']) & set(test_df['title_group'])))
print("Val-Test:", len(set(val_df['title_group']) & set(test_df['title_group'])))

Training partition: 3588
Validation partition: 906
Test partition: 615

Phone Validation: 861
Phone Test: 573

Group Overlap Check:
Train-Val: 0
Train-Test: 0
Val-Test: 0


In [16]:
train_partition = df.iloc[train_idx].copy()

train_all = train_partition.copy()
train_phone = train_partition[
    train_partition['listing_type'] == 'phone'
].copy()

print("Experiment A - All Listings")
print("Training:", len(train_all))

print("\nExperiment B - Phone Only")
print("Training:", len(train_phone))

print("\nShared Evaluation Sets")
print("Validation:", len(val_phone))
print("Test:", len(test_phone))

Experiment A - All Listings
Training: 3588

Experiment B - Phone Only
Training: 3298

Shared Evaluation Sets
Validation: 861
Test: 573


In [17]:
from sklearn.model_selection import GroupShuffleSplit

# Split validation groups for early stopping and calibration
gss3 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

early_idx, calibration_idx = next(
    gss3.split(val_df, groups=val_df['title_group'])
)

early_stop_df = val_df.iloc[early_idx].copy()
calibration_df = val_df.iloc[calibration_idx].copy()

# Keep phone listings for evaluation-related data
early_stop_phone = early_stop_df[
    early_stop_df['listing_type'] == 'phone'
].copy()

calibration_phone = calibration_df[
    calibration_df['listing_type'] == 'phone'
].copy()

# Prepare feature datasets
cat_features = [
    'model_key', 'brand', 'family',
    'condition', 'loc_area', 'loc_city'
]

def prepare_features(data):
    X = data[features].copy()
    X[cat_features] = X[cat_features].fillna('Unknown').astype(str)
    return X

X_train_all = prepare_features(train_all)
X_train_phone = prepare_features(train_phone)
X_early_stop = prepare_features(early_stop_phone)
X_test_phone = prepare_features(test_phone)

print("All Training:", X_train_all.shape)
print("Phone Training:", X_train_phone.shape)
print("Early Stopping:", X_early_stop.shape)
print("Test:", X_test_phone.shape)

print("\nMissing Categorical Values:")
print(X_train_all[cat_features].isna().sum().sum())

All Training: (3588, 22)
Phone Training: (3298, 22)
Early Stopping: (243, 22)
Test: (573, 22)

Missing Categorical Values:
0


In [19]:
print("Validation partition:", len(val_df))
print("Phone Validation:", len(val_phone))
print("Test partition:", len(test_df))
print("Phone Test:", len(test_phone))

print("\nAvailable DataFrames:")
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame) and len(obj) == 243:
        print(name, obj.shape)

Validation partition: 906
Phone Validation: 861
Test partition: 615
Phone Test: 573

Available DataFrames:
early_stop_phone (243, 66)
X_early_stop (243, 22)


In [21]:
catboost_params = {
    'iterations': 1500,
    'learning_rate': 0.03,
    'depth': 6,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'verbose': 100,
    'allow_writing_files': False
}

print(catboost_params)

{'iterations': 1500, 'learning_rate': 0.03, 'depth': 6, 'eval_metric': 'MAE', 'random_seed': 42, 'verbose': 100, 'allow_writing_files': False}


In [24]:
# Prepare categorical features consistently
train_all_X = train_all[features].copy()
train_phone_X = train_phone[features].copy()
early_stop_X = X_early_stop.copy()
test_X = test_phone[features].copy()

for col in cat_features:
    for data in [train_all_X, train_phone_X, early_stop_X, test_X]:
        data[col] = data[col].fillna("Unknown").astype(str)

print("Missing categorical values after handling:")

for name, data in {
    "All Training": train_all_X,
    "Phone Training": train_phone_X,
    "Early Stopping": early_stop_X,
    "Test": test_X
}.items():
    print(name, data[cat_features].isna().sum().sum())

Missing categorical values after handling:
All Training 0
Phone Training 0
Early Stopping 0
Test 0


In [25]:
model_all = CatBoostRegressor(
    loss_function='Quantile:alpha=0.5',
    **catboost_params
)

model_all.fit(
    train_all_X,
    train_all['price'],
    cat_features=cat_features,
    eval_set=(early_stop_X, early_stop_phone['price']),
    early_stopping_rounds=100,
    verbose=100
)

print("\nBest Iteration:", model_all.best_iteration_)
print("Best Validation MAE:", model_all.best_score_['validation']['MAE'])

0:	learn: 18116.7491729	test: 12093.0823083	best: 12093.0823083 (0)	total: 150ms	remaining: 3m 45s
100:	learn: 6395.6246087	test: 6666.0657902	best: 6658.7536650 (99)	total: 4.39s	remaining: 1m
200:	learn: 5765.1384169	test: 6367.5628324	best: 6367.5628324 (200)	total: 8.76s	remaining: 56.6s
300:	learn: 5487.9858660	test: 6238.4303774	best: 6234.5838994 (297)	total: 13.4s	remaining: 53.4s
400:	learn: 5267.1637963	test: 6272.8328630	best: 6225.5357331 (320)	total: 17.7s	remaining: 48.4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 6225.535733
bestIteration = 320

Shrink model to first 321 iterations.

Best Iteration: 320
Best Validation MAE: 6225.53573313981


In [26]:
model_phone = CatBoostRegressor(
    loss_function='Quantile:alpha=0.5',
    **catboost_params
)

model_phone.fit(
    train_phone_X,
    train_phone['price'],
    cat_features=cat_features,
    eval_set=(early_stop_X, early_stop_phone['price']),
    early_stopping_rounds=100,
    verbose=100
)

print("\nBest Iteration:", model_phone.best_iteration_)
print("Best Validation MAE:", model_phone.best_score_['validation']['MAE'])

0:	learn: 18552.3355434	test: 12245.4408257	best: 12245.4408257 (0)	total: 41.2ms	remaining: 1m 1s
100:	learn: 5960.3279411	test: 8122.0341041	best: 8120.8329696 (96)	total: 4.43s	remaining: 1m 1s
200:	learn: 5359.4257423	test: 7653.3810296	best: 7651.7774652 (198)	total: 9.09s	remaining: 58.8s
300:	learn: 5095.7795320	test: 7351.4748086	best: 7351.4748086 (300)	total: 13.8s	remaining: 54.8s
400:	learn: 4893.6152712	test: 7171.5714411	best: 7171.5013588 (399)	total: 18.3s	remaining: 50.1s
500:	learn: 4755.9713502	test: 7093.6804710	best: 7093.1916361 (498)	total: 22.5s	remaining: 44.8s
600:	learn: 4630.3531451	test: 7045.0366306	best: 7043.6174177 (599)	total: 26.6s	remaining: 39.8s
700:	learn: 4528.8152127	test: 7025.8472481	best: 7019.3827549 (634)	total: 30.8s	remaining: 35.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 7019.382755
bestIteration = 634

Shrink model to first 635 iterations.

Best Iteration: 634
Best Validation MAE: 7019.382754943587


In [27]:
from sklearn.metrics import mean_absolute_error

pred_all = model_all.predict(test_X)
pred_phone = model_phone.predict(test_X)

mae_all = mean_absolute_error(test_phone['price'], pred_all)
mae_phone = mean_absolute_error(test_phone['price'], pred_phone)

print(f"All Listings Test MAE: {mae_all:,.2f} EGP")
print(f"Phone Only Test MAE: {mae_phone:,.2f} EGP")
print(f"MAE Difference (All - Phone Only): {mae_all - mae_phone:,.2f} EGP")

All Listings Test MAE: 4,831.19 EGP
Phone Only Test MAE: 4,904.36 EGP
MAE Difference (All - Phone Only): -73.17 EGP


In [28]:
print("Target sample:")
print(test_phone['price'].head())

print("\nPrediction sample:")
print(pred_all[:5])

print("\nTarget dtype:", test_phone['price'].dtype)
print("Prediction dtype:", pred_all.dtype)

Target sample:
2     57999.0
8     19000.0
17    28000.0
21    85000.0
26    97000.0
Name: price, dtype: float64

Prediction sample:
[36477.01366058 37121.46454029 31485.79735614 25202.26454193
 86612.00915871]

Target dtype: float64
Prediction dtype: float64


In [29]:
results = pd.DataFrame({
    'Experiment': ['All Listings', 'Phone Only'],
    'Training Samples': [len(train_all), len(train_phone)],
    'Best Iteration': [
        model_all.best_iteration_,
        model_phone.best_iteration_
    ],
    'Validation MAE (EGP)': [
        model_all.best_score_['validation']['MAE'],
        model_phone.best_score_['validation']['MAE']
    ],
    'Test MAE (EGP)': [mae_all, mae_phone]
})

display(results.round(2))

improvement = mae_phone - mae_all

print(f"\nMAE Improvement: {improvement:,.2f} EGP")

if improvement > 0:
    print("Conclusion: Training on all listings achieved a lower Test MAE.")
elif improvement < 0:
    print("Conclusion: Training on phone listings only achieved a lower Test MAE.")
else:
    print("Conclusion: Both experiments achieved the same Test MAE.")

,Experiment,Training Samples,Best Iteration,Validation MAE (EGP),Test MAE (EGP)
0,All Listings,3588,320,6225.54,4831.19
1,Phone Only,3298,634,7019.38,4904.36



MAE Improvement: 73.17 EGP
Conclusion: Training on all listings achieved a lower Test MAE.
